# Heat conduction with a material inclusion

Let $\Omega=(0,3)\times(0,2)$ and $D=(1,2)\times(1/2,3/2)$. Solve

$$
\begin{align*}
-\nabla\cdot(\kappa\nabla u)=f,\qquad \kappa=\begin{cases}\eta&\text{ in }D\\1&\text{ outside }D\end{cases},\quad f=\mathbf1_D.
\end{align*}
$$

For the physical heat flux $\vec q=-\kappa\nabla u$, impose

- left: $u=0$
- right: $\vec q\cdot \vec n=\alpha(u-u_\infty)$;
- top and bottom: $\vec q\cdot \vec n=g_N$.

Use $u_\infty=g_N=0$. Positive normal flux means heat leaving the selected domain.

## Packages

Matplotlib is needed for the plots (`pip install matplotlib`), in addition to NGSolve/Netgen.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import WorkPlane, OCCGeometry, Glue, X, Y

## Geometry

The geometry contains both materials and a shared interface. Materials are named `matrix` and `inclusion`. The outer edges are `left`, `right`, `bottom`, and `top`, and the inclusion edges are `inc_left`, `inc_right`, `inc_bottom`, and `inc_top`.

In [ ]:
def inclusion_geometry():
    outer = WorkPlane().Rectangle(3, 2).Face()
    outer.edges.Min(X).name = "left"
    outer.edges.Max(X).name = "right"
    outer.edges.Min(Y).name = "bottom"
    outer.edges.Max(Y).name = "top"

    inclusion = WorkPlane().MoveTo(1, 0.5).Rectangle(1, 1).Face()
    inclusion.edges.Min(X).name = "inc_left"
    inclusion.edges.Max(X).name = "inc_right"
    inclusion.edges.Min(Y).name = "inc_bottom"
    inclusion.edges.Max(Y).name = "inc_top"
    inclusion.faces.name = "inclusion"
    matrix = outer-inclusion
    matrix.faces.name = "matrix"
    return OCCGeometry(Glue([matrix, inclusion]), dim=2)

In [ ]:
eta = 10
alpha = 1
ambient = 0
g_N = 0
order = 3
maxh = 0.2

geometry = inclusion_geometry()
mesh = Mesh(geometry.GenerateMesh(maxh=maxh))
print("Materials:", mesh.GetMaterials())
print("Boundaries:", mesh.GetBoundaries())
Draw(mesh);

## Material coefficient functions

Use `mesh.MaterialCF` for both conductivity and source. Call `material_data(mesh, eta)` again after changing `eta` or creating a new mesh.

In [ ]:
def material_data(mesh, eta):
    kappa = mesh.MaterialCF({"matrix": 1, "inclusion": eta})
    source = mesh.MaterialCF({"matrix": 0, "inclusion": 1})
    return kappa, source

kappa, source = material_data(mesh, eta)
Draw(kappa, mesh, "diffusion coefficient");

## 1. Finite element solution

Assemble the weak form, including the Robin and Neumann data. Use the given coefficients `kappa` and `source`, solve for `uh`, and display the temperature and the physical flux.

Reuse the assembly and solve pattern from [Pacman](pacman_domain.ipynb). Only `left` carries a Dirichlet condition here. Boundary integrals use `ds` instead of `dx`: `ds("right")` selects the right edge, and `ds("top|bottom")` selects both horizontal edges. For example,

```python
boundary_mass = u*v*ds("right")
boundary_load = CF(1)*v*ds("top|bottom")
```

are a boundary mass term and a unit boundary load. Insert the coefficients and signs from your weak formulation. The Robin condition contributes to the bilinear form and, for nonzero `ambient`, to the linear form. Here `ambient` represents $u_\infty$. Since `g_N` is the outward physical flux, its contribution to the linear form has a minus sign.

The left Dirichlet data are zero, so solve directly with the assembled load vector, as in Pacman. If you put the repeated solve into a function, return just `uh`.

In [ ]:
fes = H1(mesh, order=order, dirichlet="left")
u, v = fes.TnT()

# TODO: Assemble and solve the problem using kappa, source, alpha, ambient, and g_N.
# TODO: Display uh and -kappa*grad(uh).

## 2. Conductivity contrast

For `alpha=1` and `eta` in `[1, 10, 100]`, repeat the solve using `material_data(mesh, eta)`. Plot temperature profiles along `y=1` and compute the mean temperature in the inclusion. Compare the results.

To evaluate a finite element function at a physical point, use `uh(mesh(xx, yy))`. The call `mesh(xx, yy)` locates that point in the mesh. For one computed solution, sample the temperature along `y=1` using:

```python
xs = [0.01+2.98*i/100 for i in range(101)]
profile = [uh(mesh(xx, 1)) for xx in xs]
plt.plot(xs, profile, label=f"eta={eta}")
```

Use ordinary axes for these profiles. As in Pacman, call `plt.figure()` before the parameter loop, add one curve after each solve, and put the axis labels, grid, legend, and `plt.show()` after the loop. Use `"x at y=1"` and `"temperature"` as axis labels.

Select a material region with `mesh.Materials` and restrict an integral using `definedon`. The mean temperature is the temperature integral divided by the region's area:

```python
inclusion_region = mesh.Materials("inclusion")
area = Integrate(CF(1), mesh, definedon=inclusion_region)
mean_temperature = Integrate(uh, mesh, definedon=inclusion_region)/area
```

Rebuild `kappa` and `source` with `material_data(mesh, eta)` and assemble the problem again for each conductivity.

Print the mean temperature after each solve. You can append it to a simple list if you want to collect the three values.

In [ ]:
# TODO: Compare eta=1,10,100 and plot the profiles.
# TODO: Compute the mean temperature in mesh.Materials("inclusion").

## 3. Outer-edge fluxes

For `eta=10` and `alpha=1`, compute the signed outward flux through `left`, `right`, `bottom`, and `top`. Use `BoundaryFromVolumeCF` to evaluate the volume gradient on an outer edge. Compare the sum of the four fluxes with the volume source integral. On the right edge, compare the direct gradient-based flux with the Robin boundary law.

The physical flux is a volume-defined vector field. `BoundaryFromVolumeCF` evaluates it from the adjoining volume element at boundary points. `specialcf.normal(2)` gives the outward unit normal on the outer boundary. For example

```python
q = -kappa*grad(uh)
q_boundary = BoundaryFromVolumeCF(q)
n = specialcf.normal(2)
Q_left = Integrate(q_boundary*n*ds("left"), mesh)
```

The product `q_boundary*n` is the normal component. Repeat the integration for the other three outer labels. Positive values mean heat leaving the domain. Add the signed fluxes and compare their sum with `Integrate(source, mesh)` (do not sum absolute values). A small balance defect can remain for the discrete gradient-based flux.

Evaluate the Robin-law discharge separately using the temperature on the right edge

```python
Q_right_robin = Integrate(alpha*(uh-ambient)*ds("right"), mesh)
```

Compare this value with the direct right-edge flux obtained from `q_boundary`.

Start this section by setting `eta=10` and `alpha=1`, rebuilding `kappa` and `source`, and solving again.

In [ ]:
# TODO: Compute the four signed outer-edge fluxes.
# TODO: Compare their sum with the volume source integral.
# TODO: Compare the right-edge flux with the Robin boundary law.